In [1]:
!pip install spacy scikit-learn pandas -q
!python -m spacy download en_core_web_sm -q

import pandas as pd
import re
import hashlib
import spacy

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

nlp = spacy.load("en_core_web_sm")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 70.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
df = pd.read_excel('/content/edufeed_clean_data.xlsx')

df = df[df['comments'].notna() & (df['comments'] != '')]
df = df[df['sentiment_label'].notna()]
df.reset_index(drop=True, inplace=True)

In [4]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df['comments'] = df['comments'].apply(clean_text)

In [6]:
def generate_hash(text):
    return hashlib.sha256(text.encode()).hexdigest()

In [10]:
X = df['comments']
y = df['sentiment_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(
    max_features=7000,
    ngram_range=(1,3),
    min_df=2
)

X_train_vec = tfidf.fit_transform(X_train)

lr_model = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    C=2
)

lr_model.fit(X_train_vec, y_train)

LogisticRegression(C=2, class_weight='balanced', max_iter=2000)

In [11]:
def generate_hash(text):
    return hashlib.sha256(text.encode()).hexdigest()

In [12]:
def mask_text(text, name, roll, email):
    text = str(text)

    text = re.sub(r'[\w\.-]+@[\w\.-]+\.\w+', '[EMAIL]', text)

    text = text.replace(name, '[STUDENT]')
    text = text.replace(roll, '[ROLL]')
    text = text.replace(email, '[EMAIL]')

    text = re.sub(r'\b\d+\b', '[ROLL]', text)

    return text

In [13]:
def feedback_system():

    name = input("Enter your Name: ")
    roll = input("Enter your Roll No: ")
    email = input("Enter your Email: ")
    professor = input("Enter Professor Name: ")
    rating_input = float(input("Enter Rating (1-5): "))
    review = input("Enter your Review: ")

    student_id = generate_hash(name + roll + email)

    full_text = f"{name} {roll} {email} {review}"
    masked_review = mask_text(full_text, name, roll, email)

    cleaned_review = clean_text(masked_review)

    text_vec = tfidf.transform([cleaned_review])
    sentiment = lr_model.predict(text_vec)[0]

    print("\n----- OUTPUT -----")
    print("Review:", masked_review)
    print("Rating:", rating_input)
    print("Sentiment:", sentiment)

    return masked_review, rating_input, sentiment

In [22]:
feedback_system()

Enter your Name: Ram
Enter your Roll No: 86872
Enter your Email: ram@gmail.com
Enter Professor Name: prof nick
Enter Rating (1-5): 2
Enter your Review: The lectures are boring and confusing

----- OUTPUT -----
Review: [STUDENT] [ROLL] [EMAIL] The lectures are boring and confusing
Rating: 2.0
Sentiment: Negative


('[STUDENT] [ROLL] [EMAIL] The lectures are boring and confusing',
 2.0,
 'Negative')